In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os

print("Klasör dedektifi çalışıyor...\n")
for root, dirs, files in os.walk('/kaggle/input'):
    # İçinde hastalık isimleri olan (10'dan fazla alt klasörü olan) ana klasörleri bulur
    if len(dirs) > 10:
        print("BULDUM:", root)

Klasör dedektifi çalışıyor...

BULDUM: /kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/test
BULDUM: /kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/train
BULDUM: /kaggle/input/datasets/emmarex/plantdisease/PlantVillage
BULDUM: /kaggle/input/datasets/emmarex/plantdisease/plantvillage/PlantVillage


In [3]:
import os, shutil, random, re

def clean_name(name):
    # İsimleri Keras'ın anlayacağı standart formata çevirir
    return '_'.join(re.sub(r'[^a-z0-9]', ' ', name.lower()).split())

# SENİN BULDUĞUN KESİN DOSYA YOLLARI
pv_dir = '/kaggle/input/datasets/emmarex/plantdisease/PlantVillage'
pd_train_dir = '/kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/train'
pd_test_dir = '/kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/test'

# Kaggle'da oluşturacağımız çalışma klasörleri
hybrid_dir = '/kaggle/working/Hybrid_Dataset'
hybrid_train = os.path.join(hybrid_dir, 'train')
hybrid_val = os.path.join(hybrid_dir, 'val')

print("Hedef klasörler oluşturuluyor ve veriler kopyalanıyor...")
print("Bu işlem P100 GPU üzerinde bile birkaç dakika sürebilir, lütfen bekleyin...\n")

# Klasörleri baştan yaratıyoruz (Hatanın kesin çözümü)
os.makedirs(hybrid_train, exist_ok=True)
os.makedirs(hybrid_val, exist_ok=True)

# 1. PlantVillage verilerini kopyala (%80 Train, %20 Val)
if os.path.exists(pv_dir):
    for cls in os.listdir(pv_dir):
        cls_path = os.path.join(pv_dir, cls)
        if os.path.isdir(cls_path):
            c_name = clean_name(cls)
            os.makedirs(os.path.join(hybrid_train, c_name), exist_ok=True)
            os.makedirs(os.path.join(hybrid_val, c_name), exist_ok=True)
            imgs = os.listdir(cls_path)
            random.shuffle(imgs)
            split = int(len(imgs) * 0.8)
            for img in imgs[:split]: shutil.copy(os.path.join(cls_path, img), os.path.join(hybrid_train, c_name, img))
            for img in imgs[split:]: shutil.copy(os.path.join(cls_path, img), os.path.join(hybrid_val, c_name, img))

# 2. PlantDoc Train verilerini kopyala
if os.path.exists(pd_train_dir):
    for cls in os.listdir(pd_train_dir):
        cls_path = os.path.join(pd_train_dir, cls)
        if os.path.isdir(cls_path):
            c_name = clean_name(cls)
            os.makedirs(os.path.join(hybrid_train, c_name), exist_ok=True)
            for img in os.listdir(cls_path):
                shutil.copy(os.path.join(cls_path, img), os.path.join(hybrid_train, c_name, f"pd_{img}"))

# 3. PlantDoc Test verilerini kopyala (Val için)
if os.path.exists(pd_test_dir):
    for cls in os.listdir(pd_test_dir):
        cls_path = os.path.join(pd_test_dir, cls)
        if os.path.isdir(cls_path):
            c_name = clean_name(cls)
            os.makedirs(os.path.join(hybrid_val, c_name), exist_ok=True)
            for img in os.listdir(cls_path):
                shutil.copy(os.path.join(cls_path, img), os.path.join(hybrid_val, c_name, f"pd_{img}"))

print("Tüm veriler başarıyla kopyalandı!")
print("Eğitim setindeki gerçek tarla verileri (PlantDoc) 15 katına çıkarılıyor...")

for cls in os.listdir(hybrid_train):
    cls_path = os.path.join(hybrid_train, cls)
    if os.path.isdir(cls_path):
        pd_imgs = [img for img in os.listdir(cls_path) if img.startswith('pd_') and not img.endswith('copy.jpg')]
        for img in pd_imgs:
            orig = os.path.join(cls_path, img)
            for i in range(15):
                shutil.copy(orig, os.path.join(cls_path, f"{img.split('.')[0]}_copy{i}.jpg"))

print("\nHarika! Kaggle için Hibrit Veri Seti Başarıyla Hazırlandı!")

Hedef klasörler oluşturuluyor ve veriler kopyalanıyor...
Bu işlem P100 GPU üzerinde bile birkaç dakika sürebilir, lütfen bekleyin...

Tüm veriler başarıyla kopyalandı!
Eğitim setindeki gerçek tarla verileri (PlantDoc) 15 katına çıkarılıyor...

Harika! Kaggle için Hibrit Veri Seti Başarıyla Hazırlandı!


In [5]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print("Jeneratörler ayarlanıyor...")
train_datagen = ImageDataGenerator(
    rotation_range=30, width_shift_range=0.2, height_shift_range=0.2,
    shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest'
)
val_datagen = ImageDataGenerator()

train_generator = train_datagen.flow_from_directory('/kaggle/working/Hybrid_Dataset/train', target_size=(224, 224), batch_size=32, class_mode='categorical')
val_generator = val_datagen.flow_from_directory('/kaggle/working/Hybrid_Dataset/val', target_size=(224, 224), batch_size=32, class_mode='categorical')

# Tüm Kaggle klasörlerini tara ve modelini kesin olarak bul
model_path = ""
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.keras') or file.endswith('.h5'):
            model_path = os.path.join(root, file)
            break
    if model_path: # Modeli bulduysa aramayı kes
        break

if not model_path:
    print("HATA: .keras veya .h5 uzantılı model dosyası bulunamadı! Sağ panelden yüklendiğine emin ol.")
else:
    print(f"BULDUM! Drive'dan gelen model şuradan yükleniyor: {model_path}")
    model = load_model(model_path)

    # Fine-Tuning için kilidi aç
    model.trainable = True
    model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

    # Yeni rekorları Kaggle'ın Output klasörüne kaydedeceğiz
    checkpoint_path = '/kaggle/working/best_hybrid_model_finetuned.keras'
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        ModelCheckpoint(filepath=checkpoint_path, monitor='val_accuracy', save_best_only=True)
    ]

    print("Kaggle P100 GPU aktif! İnce Ayar Eğitimi Başlıyor...")
    history_fine = model.fit(train_generator, epochs=10, validation_data=val_generator, callbacks=callbacks)

Jeneratörler ayarlanıyor...
Found 53530 images belonging to 41 classes.
Found 4370 images belonging to 41 classes.
BULDUM! Drive'dan gelen model şuradan yükleniyor: /kaggle/input/datasets/erenogut/bitki-hastalik/best_hybrid_model.keras


I0000 00:00:1771493985.294395      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Kaggle P100 GPU aktif! İnce Ayar Eğitimi Başlıyor...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1771494001.358608     160 service.cc:152] XLA service 0x7f65dc110b90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1771494001.358649     160 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1771494003.812875     160 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1771494015.235237     160 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1673/1673 ━━━━━━━━━━━━━━━━━━━━ 1206s 706ms/step - accuracy: 0.9344 - loss: 0.2401 - val_accuracy: 0.9087 - val_loss: 0.2994
Epoch 2/10
1673/1673 ━━━━━━━━━━━━━━━━━━━━ 1098s 656ms/step - accuracy: 0.9340 - loss: 0.2403 - val_accuracy: 0.9082 - val_loss: 0.2981
Epoch 3/10
1673/1673 ━━━━━━━━━━━━━━━━━━━━ 1062s 635ms/step - accuracy: 0.9347 - loss: 0.2414 - val_accuracy: 0.9078 - val_loss: 0.2982
Epoch 4/10
1673/1673 ━━━━━━━━━━━━━━━━━━━━ 1032s 617ms/step - accuracy: 0.9340 - loss: 0.2372 - val_accuracy: 0.9087 - val_loss: 0.2960
Epoch 5/10
1673/1673 ━━━━━━━━━━━━━━━━━━━━ 1023s 611ms/step - accuracy: 0.9341 - loss: 0.2361 - val_accuracy: 0.9071 - val_loss: 0.2976
Epoch 6/10
1673/1673 ━━━━━━━━━━━━━━━━━━━━ 1015s 607ms/step - accuracy: 0.9360 - loss: 0.2361 - val_accuracy: 0.9080 - val_loss: 0.2959
Epoch 7/10
1673/1673 ━━━━━━━━━━━━━━━━━━━━ 1073s 641ms/step - accuracy: 0.9353 - loss: 0.2385 - val_accuracy: 0.9089 - val_loss: 0.2948
Epoch 8/10
1673/1673 ━━━━━━━━━━━━━━━━━━━━ 1105s 660ms/step - accur

In [2]:
import os
import re

def clean_name(name):
    # İsimleri eğitimdeki formatla birebir aynı yapmak için temizliyoruz
    return '_'.join(re.sub(r'[^a-z0-9]', ' ', name.lower()).split())

# Sabit Kaggle Input Yolları (Asla silinmezler)
pv_dir = '/kaggle/input/datasets/emmarex/plantdisease/PlantVillage'
pd_train_dir = '/kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset/train'

siniflar = set() # Tekrar eden isimleri engellemek için küme (set) kullanıyoruz

# 1. PlantVillage sınıflarını topla
if os.path.exists(pv_dir):
    for cls in os.listdir(pv_dir):
        if os.path.isdir(os.path.join(pv_dir, cls)):
            siniflar.add(clean_name(cls))

# 2. PlantDoc sınıflarını topla
if os.path.exists(pd_train_dir):
    for cls in os.listdir(pd_train_dir):
        if os.path.isdir(os.path.join(pd_train_dir, cls)):
            siniflar.add(clean_name(cls))

# Keras ImageDataGenerator her zaman klasörleri alfabetik sıraya dizer
sirali_siniflar = sorted(list(siniflar))

# Backend'in anlayacağı Sözlük (Dictionary) formatına çevir
class_indices = {i: sinif for i, sinif in enumerate(sirali_siniflar)}

print("Backend'ci arkadaşa gönderilecek Sınıf Listesi (Class Indices):\n")
print(class_indices)

Backend'ci arkadaşa gönderilecek Sınıf Listesi (Class Indices):

{0: 'apple_leaf', 1: 'apple_rust_leaf', 2: 'apple_scab_leaf', 3: 'bell_pepper_leaf', 4: 'bell_pepper_leaf_spot', 5: 'blueberry_leaf', 6: 'cherry_leaf', 7: 'corn_gray_leaf_spot', 8: 'corn_leaf_blight', 9: 'corn_rust_leaf', 10: 'grape_leaf', 11: 'grape_leaf_black_rot', 12: 'peach_leaf', 13: 'pepper_bell_bacterial_spot', 14: 'pepper_bell_healthy', 15: 'potato_early_blight', 16: 'potato_healthy', 17: 'potato_late_blight', 18: 'potato_leaf_early_blight', 19: 'potato_leaf_late_blight', 20: 'raspberry_leaf', 21: 'soyabean_leaf', 22: 'squash_powdery_mildew_leaf', 23: 'strawberry_leaf', 24: 'tomato_bacterial_spot', 25: 'tomato_early_blight', 26: 'tomato_early_blight_leaf', 27: 'tomato_healthy', 28: 'tomato_late_blight', 29: 'tomato_leaf', 30: 'tomato_leaf_bacterial_spot', 31: 'tomato_leaf_late_blight', 32: 'tomato_leaf_mold', 33: 'tomato_leaf_mosaic_virus', 34: 'tomato_leaf_yellow_virus', 35: 'tomato_mold_leaf', 36: 'tomato_septor